In [1]:
from cirro import DataPortal

portal = DataPortal()
helper = portal.developer_helper

In [5]:
import json
from cirro.helpers.preprocess_dataset import PreprocessDataset

WORKFLOW_PREFIX = "Mutect2"

# ------------------------
# Functions (no mock dataset)
# ------------------------
def yield_single_inputs(ds: PreprocessDataset):
    df = ds.files  # this is a DataFrame with ["sample", "file"]
    for base_name, group in df.groupby("sample"):
        if "PBMC" in base_name:
            continue

        sample_to_analyze = None
        sample_to_analyze_index = None
        normal_bam = None
        normal_bai = None

        for f in group["file"]:
            if f.endswith(".bam") and not f.endswith(".bam.bai"):
                if "PBMC" in base_name:
                    normal_bam = f
                else:
                    sample_to_analyze = f
            elif f.endswith(".bam.bai"):
                if "PBMC" in base_name:
                    normal_bai = f
                else:
                    sample_to_analyze_index = f

        if sample_to_analyze and sample_to_analyze_index:
            yield {
                f"{WORKFLOW_PREFIX}.tumor_reads": sample_to_analyze,
                f"{WORKFLOW_PREFIX}.tumor_reads_index": sample_to_analyze_index,
            }
        
        if normal_bam and normal_bai:
            yield {
                f"{WORKFLOW_PREFIX}.normal_reads": normal_bam,
                f"{WORKFLOW_PREFIX}.normal_reads_index": normal_bai,
            }


def collapse_arrays(obj):
    if isinstance(obj, list):
        return [collapse_arrays(o) for o in obj]
    elif isinstance(obj, dict):
        new = {}
        for k, v in obj.items():
            if isinstance(v, list) and len(v) == 1:
                new[k] = v[0]
            else:
                new[k] = v
        return new
    else:
        return obj


def setup_inputs(ds: PreprocessDataset):
    all_inputs = [
        {
            **single_input,
            **{
                kw: val
                for kw, val in ds.params.items()
                if kw.startswith(WORKFLOW_PREFIX)
            },
        }
        for single_input in yield_single_inputs(ds)
    ]
    assert len(all_inputs) > 0, "No inputs found -- stopping execution"
    return collapse_arrays(all_inputs)


def setup_options(ds: PreprocessDataset):
    # Add script bucket name from out_dir
    ds.add_param("scriptBucketName", ds.params["out_dir"].split("/")[2])
    options = {
        kw: val
        for kw, val in ds.params.items()
        if not kw.startswith(WORKFLOW_PREFIX)
    }
    return collapse_arrays(options)


# ------------------------
# Use real dataset
# ------------------------
dataset = portal.get_dataset(
    project="BTC-GBM-Production",
    dataset="Aligning P4 fastqs to generate BAMs rerun"
)

files = dataset.list_files()
selected_files = [f for f in files if "GBM1.DFCI4.S1" in str(f.name) and f.name.endswith(".bam")]
selected_files += [f for f in files if "GBM1.DFCI4.S1" in str(f.name) and f.name.endswith(".bai")]

normal_files = [f for f in files if "GBM1.DFCI4.PBMC" in str(f.name) and f.name.endswith(".bam")]
normal_files += [f for f in files if "GBM1.DFCI4.PBMC" in str(f.name) and f.name.endswith(".bai")]

chosen_files = selected_files + normal_files

ds = helper.generate_preprocess_for_input_datasets(
    project_id=dataset.project_id,
    input_dataset_ids=[f.id for f in chosen_files]
)

# Now run setup functions
inputs = setup_inputs(ds)
options = setup_options(ds)

print("Inputs:", json.dumps(inputs, indent=2))
print("Options:", json.dumps(options, indent=2))


AttributeError: 'NoneType' object has no attribute 'samples'